In [ ]:
import os
import shutil
import datetime
from google.colab import drive

# Installer tqdm si nécessaire
try:
    from tqdm import tqdm
    print("✅ tqdm est déjà installé")
except ImportError:
    print("⏳ Installation de tqdm...")
    !pip install -q tqdm
    from tqdm import tqdm
    print("✅ tqdm installé avec succès")

# Monter Google Drive
drive.mount('/content/drive')

# ================= CONFIGURATION =================
parent_path = "/content/drive/MyDrive/plantes_dataset"
dest_path = os.path.join(parent_path, "Dataset_Fusion")

feuille_classes = ["Brulure", "Rouille_Commune", "Sain", "Tache_grise"]
tige_classes = ["Pourriture_charbonnee", "Pourriture_petiole", "Tige_saine"]

os.makedirs(dest_path, exist_ok=True)
all_classes = feuille_classes + tige_classes
for cls in all_classes:
    os.makedirs(os.path.join(dest_path, cls), exist_ok=True)

IMAGE_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.bmp', '.gif', '.tiff', '.webp')

def move_files(src_dir, dest_dir, prefix=""):
    """Déplace les fichiers avec gestion d'erreurs et barre de progression"""
    if not os.path.exists(src_dir):
        print(f"⚠️ Dossier introuvable: {src_dir}")
        return 0

    files = [f for f in os.listdir(src_dir)
             if os.path.isfile(os.path.join(src_dir, f)) and
             f.lower().endswith(IMAGE_EXTENSIONS)]

    if not files:
        print(f"⚠️ Aucune image trouvée dans: {src_dir}")
        return 0

    count = 0
    for file in tqdm(files, desc=f"Déplacement {os.path.basename(src_dir)}", unit="file"):
        src_file = os.path.join(src_dir, file)
        dest_file = os.path.join(dest_dir, f"{prefix}{file}")

        try:
            shutil.move(src_file, dest_file)
            count += 1
        except Exception as e:
            print(f"❌ Erreur lors du déplacement de {src_file}: {e}")

    return count

# ================ FUSION FEUILLES ================
print("\n" + "="*50)
print("TRAITEMENT DES FEUILLES")
print("="*50)

dataset1_path = os.path.join(parent_path, "dataset_1")
if os.path.exists(dataset1_path):
    print("\n> Traitement de dataset_1")
    for cls in feuille_classes:
        src_dir = os.path.join(dataset1_path, cls)
        dest_dir = os.path.join(dest_path, cls)
        moved = move_files(src_dir, dest_dir, prefix="D1_")
        print(f"  - {cls}: {moved} images déplacées")

for dataset_name in ["dataset_2", "dataset_3"]:
    dataset_path = os.path.join(parent_path, dataset_name)
    print(f"\n> Traitement de {dataset_name}")
    if os.path.exists(dataset_path):
        for split in ["Train", "Valid", "Test"]:
            split_path = os.path.join(dataset_path, split)
            if os.path.exists(split_path):
                print(f"  - Split: {split}")
                for cls in feuille_classes:
                    src_dir = os.path.join(split_path, cls)
                    dest_dir = os.path.join(dest_path, cls)
                    dataset_id = dataset_name.split("_")[1]
                    prefix = f"D{dataset_id}_{split}_"
                    moved = move_files(src_dir, dest_dir, prefix=prefix)
                    print(f"    - {cls}: {moved} images déplacées")

# ================ FUSION TIGES ================
print("\n" + "="*50)
print("TRAITEMENT DES TIGES")
print("="*50)

for cls_name in tige_classes:
    print(f"\n> Traitement des {cls_name}")
    src_dir = os.path.join(parent_path, cls_name, cls_name)
    dest_dir = os.path.join(dest_path, cls_name)
    if not os.path.exists(src_dir):
        print(f"⚠️ Chemin manquant: {src_dir}")
        continue
    moved = move_files(src_dir, dest_dir, prefix="Tige_")
    print(f"  - {cls_name}: {moved} images déplacées")

# ================ RAPPORT FINAL ================
print("\n" + "="*50)
print("RÉCAPITULATIF")
print("="*50)

class_counts = {}
total_images = 0
for cls in all_classes:
    cls_path = os.path.join(dest_path, cls)
    images = [f for f in os.listdir(cls_path) if f.lower().endswith(IMAGE_EXTENSIONS)]
    count = len(images)
    class_counts[cls] = count
    total_images += count

for cls, count in class_counts.items():
    print(f"- {cls}: {count} images")

print(f"\nTOTAL: {total_images} images déplacées")

report_path = os.path.join(dest_path, "rapport_fusion.txt")
with open(report_path, "w", encoding='utf-8') as f:
    f.write("RAPPORT DE FUSION\n")
    f.write("="*40 + "\n")
    f.write(f"Date: {datetime.datetime.now()}\n")
    f.write(f"Source: {parent_path}\nDestination: {dest_path}\n\n")
    f.write("CLASSES :\n")
    for cls, count in class_counts.items():
        f.write(f"{cls.ljust(25)}: {count} images\n")
    f.write(f"\nTOTAL: {total_images} images\n")

print(f"\n✅ Rapport généré: {report_path}")
print("\n✅ Fusion terminée (déplacement effectué sans duplication).")


In [ ]:
import os
import shutil

# Chemins
src_dir = "/content/drive/MyDrive/plantes_dataset/Pourriture_charbonnee/Pourriture_charbonnee"
dest_dir = "/content/drive/MyDrive/plantes_dataset/Dataset_Fusion/Pourriture_charbonnee"

# Vérification
if not os.path.exists(src_dir):
    print("❌ Dossier source introuvable :", src_dir)
else:
    print("✅ Dossier source trouvé, déplacement en cours...")

    moved = 0
    for file in os.listdir(src_dir):
        if file.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.gif', '.tiff', '.webp')):
            src_file = os.path.join(src_dir, file)
            dest_file = os.path.join(dest_dir, f"Tige_{file}")
            try:
                shutil.move(src_file, dest_file)
                moved += 1
            except Exception as e:
                print(f"❌ Erreur pour {file} : {e}")

    print(f"\n✅ {moved} images déplacées vers Pourriture_charbonnee.")


In [ ]:
import os

test_path = "/content/drive/MyDrive/plantes_dataset/dataset_1/Brulure"

# Afficher les 5 premières images
print(os.listdir(test_path)[:5])


In [ ]:
import os
import datetime

# Chemins
dest_path = "/content/drive/MyDrive/plantes_dataset/Dataset_Fusion"
report_path = os.path.join(dest_path, "rapport_fusion.txt")

# Classes à récapituler
feuille_classes = ["Brulure", "Rouille_Commune", "Sain", "Tache_grise"]
tige_classes = ["Pourriture_charbonnee", "Pourriture_petiole", "Tige_saine"]
all_classes = feuille_classes + tige_classes

# Extensions valides
IMAGE_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.bmp', '.gif', '.tiff', '.webp')

# Comptage par classe
class_counts = {}
total_images = 0

for cls in all_classes:
    cls_path = os.path.join(dest_path, cls)
    images = [f for f in os.listdir(cls_path) if f.lower().endswith(IMAGE_EXTENSIONS)]
    count = len(images)
    class_counts[cls] = count
    total_images += count

# Écriture du rapport
with open(report_path, "w", encoding="utf-8") as f:
    f.write("RAPPORT DE FUSION DE DATASET\n")
    f.write("="*50 + "\n\n")
    f.write(f"Date : {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"Dossier de destination : {dest_path}\n\n")
    f.write("RÉCAPITULATIF PAR CLASSE\n")
    f.write("-" * 50 + "\n")
    for cls in all_classes:
        f.write(f"{cls.ljust(25)} : {class_counts[cls]} images\n")
    f.write("\n" + "="*50 + "\n")
    f.write(f"TOTAL IMAGES : {total_images}\n")

print("✅ Rapport mis à jour avec succès !")


In [ ]:
import os
import shutil
import random
from google.colab import drive

# Monter Google Drive
drive.mount('/content/drive')

# Base
base_path = "/content/drive/MyDrive/plantes_dataset/Dataset_Fusion"

# Classes existantes
classes = ["Brulure", "Rouille_Commune", "Sain", "Tache_grise",
           "Pourriture_charbonnee", "Pourriture_petiole", "Tige_saine"]

# Pourcentages
ratios = {
    "Train": 0.8,
    "Valid": 0.1,
    "Test": 0.1
}

# Extensions valides
valid_ext = (".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".webp")

# Vérification des ratios
total_ratio = sum(ratios.values())
if abs(total_ratio - 1.0) > 0.01:  # Tolérance de 1%
    print(f"⚠️ ATTENTION: Les ratios ({total_ratio}) ne somment pas à 1. Ajustement automatique...")
    # Ajustement des ratios
    for key in ratios:
        ratios[key] /= total_ratio

# Vérification du chemin de base
if not os.path.exists(base_path):
    print(f"❌ ERREUR: Le chemin de base '{base_path}' n'existe pas!")
    exit()

# Traitement
for cls in classes:
    print(f"\n📂 Traitement de la classe : {cls}")
    class_path = os.path.join(base_path, cls)

    # Vérifier si le dossier de classe existe
    if not os.path.exists(class_path):
        print(f"  ⚠️ Dossier introuvable: {class_path}")
        continue

    # Lister uniquement les fichiers images
    images = [f for f in os.listdir(class_path)
              if os.path.isfile(os.path.join(class_path, f)) and
              f.lower().endswith(valid_ext)]

    # Vérifier s'il y a des images
    if not images:
        print(f"  ⚠️ Aucune image trouvée dans: {class_path}")
        continue

    # Afficher le nombre d'images
    print(f"  - Total d'images: {len(images)}")

    # Mélanger les images
    random.shuffle(images)

    # Calculer les indices de séparation
    total = len(images)
    train_end = int(ratios["Train"] * total)
    valid_end = train_end + int(ratios["Valid"] * total)

    # Créer les splits
    splits = {
        "Train": images[:train_end],
        "Valid": images[train_end:valid_end],
        "Test": images[valid_end:]
    }

    # Créer les dossiers et déplacer les fichiers
    for split, split_imgs in splits.items():
        split_dir = os.path.join(class_path, split)
        os.makedirs(split_dir, exist_ok=True)

        for img in split_imgs:
            src = os.path.join(class_path, img)
            dst = os.path.join(split_dir, img)

            # Vérifier si le fichier source existe
            if not os.path.exists(src):
                print(f"    ⚠️ Fichier introuvable: {src}")
                continue

            try:
                shutil.move(src, dst)
            except Exception as e:
                print(f"    ❌ Erreur lors du déplacement de {src}: {str(e)}")

        print(f"  - {split}: {len(split_imgs)} images déplacées")

print("\n✅ Division terminée dans le dossier Dataset_Fusion.")

In [ ]:
import os
import shutil
from google.colab import drive

# Monter Google Drive
drive.mount('/content/drive')

# Chemins de base
base_path = "/content/drive/MyDrive/plantes_dataset/Dataset_Fusion"
organized_path = "/content/drive/MyDrive/plantes_dataset/Dataset_Organized"

# Créer la structure organisée
for folder in ["Train", "Valid", "Test"]:
    os.makedirs(os.path.join(organized_path, folder), exist_ok=True)
    for cls in ["Brulure", "Rouille_Commune", "Sain", "Tache_grise",
                "Pourriture_charbonnee", "Pourriture_petiole", "Tige_saine"]:
        os.makedirs(os.path.join(organized_path, folder, cls), exist_ok=True)

# Fonction pour réorganiser les fichiers
def organize_dataset():
    print("Réorganisation du dataset en cours...")

    # Pour chaque classe
    classes = ["Brulure", "Rouille_Commune", "Sain", "Tache_grise",
               "Pourriture_charbonnee", "Pourriture_petiole", "Tige_saine"]

    for cls in classes:
        print(f"\nTraitement de la classe: {cls}")
        cls_path = os.path.join(base_path, cls)

        # Déplacer les fichiers d'entraînement
        train_src = os.path.join(cls_path, "Train")
        train_dest = os.path.join(organized_path, "Train", cls)
        if os.path.exists(train_src):
            for file in os.listdir(train_src):
                src_file = os.path.join(train_src, file)
                dest_file = os.path.join(train_dest, file)
                shutil.move(src_file, dest_file)
            print(f"  - Fichiers Train déplacés: {len(os.listdir(train_dest))}")

        # Déplacer les fichiers de validation
        valid_src = os.path.join(cls_path, "Valid")
        valid_dest = os.path.join(organized_path, "Valid", cls)
        if os.path.exists(valid_src):
            for file in os.listdir(valid_src):
                src_file = os.path.join(valid_src, file)
                dest_file = os.path.join(valid_dest, file)
                shutil.move(src_file, dest_file)
            print(f"  - Fichiers Valid déplacés: {len(os.listdir(valid_dest))}")

        # Déplacer les fichiers de test
        test_src = os.path.join(cls_path, "Test")
        test_dest = os.path.join(organized_path, "Test", cls)
        if os.path.exists(test_src):
            for file in os.listdir(test_src):
                src_file = os.path.join(test_src, file)
                dest_file = os.path.join(test_dest, file)
                shutil.move(src_file, dest_file)
            print(f"  - Fichiers Test déplacés: {len(os.listdir(test_dest))}")

    print("\n✅ Réorganisation terminée avec succès!")

# Exécuter la réorganisation
organize_dataset()

# Vérification finale
print("\nStructure finale:")
print(f"Train: {len(os.listdir(os.path.join(organized_path, 'Train')))} classes")
print(f"Valid: {len(os.listdir(os.path.join(organized_path, 'Valid')))} classes")
print(f"Test: {len(os.listdir(os.path.join(organized_path, 'Test')))} classes")

for cls in ["Brulure", "Rouille_Commune", "Sain"]:
    train_cls_path = os.path.join(organized_path, "Train", cls)
    print(f"\nClasse {cls}:")
    print(f"  Train: {len(os.listdir(train_cls_path))} images")
    print(f"  Valid: {len(os.listdir(os.path.join(organized_path, 'Valid', cls)))} images")
    print(f"  Test: {len(os.listdir(os.path.join(organized_path, 'Test', cls)))} images")

In [ ]:
# Affichage pour toutes les classes
classes = ["Brulure", "Rouille_Commune", "Sain", "Tache_grise",
           "Pourriture_charbonnee", "Pourriture_petiole", "Tige_saine"]

for cls in classes:
    print(f"\nClasse {cls}:")
    for split in ["Train", "Valid", "Test"]:
        path = os.path.join(organized_path, split, cls)
        count = len(os.listdir(path)) if os.path.exists(path) else 0
        print(f"  {split}: {count} images")


In [ ]:
%%time
# ========== CONFIGURATION INITIALE ==========
import os
import shutil
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
from PIL import Image, UnidentifiedImageError
from google.colab import drive
import datetime

# ========== MONTER GOOGLE DRIVE ==========
drive.mount('/content/drive')

# ========== PARAMÈTRES ==========
# Chemins
base_path = "/content/drive/MyDrive/plantes_dataset"
organized_path = os.path.join(base_path, "Dataset_Organized")
corrupted_path = os.path.join(base_path, "Corrupted_Images")

# Chemins des ensembles
train_dir = os.path.join(organized_path, "Train")
valid_dir = os.path.join(organized_path, "Valid")
test_dir = os.path.join(organized_path, "Test")

# Paramètres d'entraînement
img_size = (224, 224)
batch_size = 32
epochs = 50
expected_classes = ["Brulure", "Rouille_Commune", "Sain", "Tache_grise",
                    "Pourriture_charbonnee", "Pourriture_petiole", "Tige_saine"]

# ========== SAUVEGARDE DES IMAGES CORROMPUES ==========
print("="*50)
print("GESTION DES IMAGES CORROMPUES")
print("="*50)

# Créer le dossier pour les images corrompues
os.makedirs(corrupted_path, exist_ok=True)
print(f"Dossier des images corrompues créé: {corrupted_path}")

def move_corrupted_images(src_dir, dest_base_dir):
    """Déplace les images corrompues vers un dossier spécial"""
    corrupted_files = []
    for root, _, files in os.walk(src_dir):
        for file in files:
            file_path = os.path.join(root, file)
            try:
                # Vérifier les extensions valides
                if file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif', '.tiff', '.webp')):
                    # Vérifier la validité de l'image
                    with Image.open(file_path) as img:
                        img.verify()
            except (IOError, SyntaxError, UnidentifiedImageError, OSError) as e:
                # Créer une structure de dossiers similaire pour les corrompues
                relative_path = os.path.relpath(root, src_dir)
                dest_dir = os.path.join(dest_base_dir, relative_path)
                os.makedirs(dest_dir, exist_ok=True)

                # Déplacer l'image corrompue
                dest_path = os.path.join(dest_dir, file)
                shutil.move(file_path, dest_path)
                corrupted_files.append((file_path, dest_path))
    return corrupted_files

# Déplacer les images corrompues
print("\nTraitement du dossier Train...")
corrupted_train = move_corrupted_images(train_dir, os.path.join(corrupted_path, "Train"))
print(f"  - {len(corrupted_train)} images corrompues déplacées")

print("\nTraitement du dossier Valid...")
corrupted_valid = move_corrupted_images(valid_dir, os.path.join(corrupted_path, "Valid"))
print(f"  - {len(corrupted_valid)} images corrompues déplacées")

print("\nTraitement du dossier Test...")
corrupted_test = move_corrupted_images(test_dir, os.path.join(corrupted_path, "Test"))
print(f"  - {len(corrupted_test)} images corrompues déplacées")

# Créer un rapport des images corrompues
corrupted_report = os.path.join(corrupted_path, "corrupted_images_report.txt")
with open(corrupted_report, "w") as f:
    f.write("Rapport des Images Corrompues\n")
    f.write("="*50 + "\n\n")
    f.write(f"Date: {datetime.datetime.now()}\n")
    f.write(f"Total d'images corrompues: {len(corrupted_train) + len(corrupted_valid) + len(corrupted_test)}\n\n")

    f.write("Images corrompues dans Train:\n")
    for src, dest in corrupted_train:
        f.write(f"- Origine: {src}\n  Déplacée à: {dest}\n")

    f.write("\nImages corrompues dans Valid:\n")
    for src, dest in corrupted_valid:
        f.write(f"- Origine: {src}\n  Déplacée à: {dest}\n")

    f.write("\nImages corrompues dans Test:\n")
    for src, dest in corrupted_test:
        f.write(f"- Origine: {src}\n  Déplacée à: {dest}\n")

print(f"\n✅ Rapport des images corrompues sauvegardé: {corrupted_report}")

# ========== PRÉPARATION DES DONNÉES ==========
print("\n" + "="*50)
print("PRÉPARATION DES DONNÉES")
print("="*50)

# Fonction pour vérifier les fichiers valides
def is_valid_image(filename):
    valid_ext = ('.png', '.jpg', '.jpeg', '.bmp', '.gif', '.tiff', '.webp', '.jfif')
    return filename.lower().endswith(valid_ext) and not filename.startswith('.')

# Générateurs d'images
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

test_datagen = ImageDataGenerator(rescale=1./255)

# Création des générateurs avec vérification des fichiers
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=True,
    seed=42,
)

validation_generator = test_datagen.flow_from_directory(
    valid_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False,
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False,
)

# Vérification des classes
class_names = list(train_generator.class_indices.keys())
num_classes = len(class_names)
print("\nClasses détectées:", class_names)
print(f"Nombre de classes: {num_classes}")

# ========== CONSTRUCTION DU MODÈLE ==========
print("\n" + "="*50)
print("CONSTRUCTION DU MODÈLE")
print("="*50)

model = Sequential([
    Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(img_size[0], img_size[1], 3)),
    BatchNormalization(),
    Conv2D(32, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    Dropout(0.25),

    Conv2D(64, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    Conv2D(64, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    Dropout(0.25),

    Conv2D(128, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    Conv2D(128, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    Dropout(0.25),

    Flatten(),
    Dense(512, activation='relu'),
    BatchNormalization(),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# ========== CALLBACKS ==========
callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=0.00001),
    ModelCheckpoint(
        filepath='/content/drive/MyDrive/plantes_dataset/best_model.h5',
        monitor='val_accuracy',
        save_best_only=True,
        save_weights_only=False,
        mode='max',
        verbose=1
    )
]

# ========== ENTRAÎNEMENT ==========
print("\n" + "="*50)
print("DÉBUT DE L'ENTRAÎNEMENT")
print("="*50)

# Calcul des steps avec gestion des petites tailles
steps_per_epoch = max(1, train_generator.samples // batch_size)
validation_steps = max(1, validation_generator.samples // batch_size)

history = model.fit(
    train_generator,
    steps_per_epoch=steps_per_epoch,
    epochs=epochs,
    validation_data=validation_generator,
    validation_steps=validation_steps,
    callbacks=callbacks
)

# ========== ÉVALUATION ==========
print("\n" + "="*50)
print("ÉVALUATION DU MODÈLE")
print("="*50)

# Charger le meilleur modèle
model = tf.keras.models.load_model('/content/drive/MyDrive/plantes_dataset/best_model.h5')

# Évaluation sur le test set
test_loss, test_acc = model.evaluate(test_generator)
print(f"\nPrécision sur le test set: {test_acc:.4f}")
print(f"Perte sur le test set: {test_loss:.4f}")

# Prédictions
y_pred = model.predict(test_generator)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = test_generator.classes

# Rapport de classification
print("\nRapport de classification:")
print(classification_report(y_true, y_pred_classes, target_names=class_names))

# Matrice de confusion
cm = confusion_matrix(y_true, y_pred_classes)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Prédictions')
plt.ylabel('Vraies étiquettes')
plt.title('Matrice de Confusion')
plt.savefig('/content/drive/MyDrive/plantes_dataset/confusion_matrix.png')
plt.show()

# ========== VISUALISATION ==========
print("\n" + "="*50)
print("VISUALISATION DES RÉSULTATS")
print("="*50)

# Courbes d'apprentissage
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/plantes_dataset/training_history.png')
plt.show()

# Exemples de prédictions
plt.figure(figsize=(15, 15))
images, labels = next(test_generator)
for i in range(9):
    plt.subplot(3, 3, i + 1)
    img = images[i]
    true_idx = np.argmax(labels[i])
    true_label = class_names[true_idx]

    pred = model.predict(np.expand_dims(img, axis=0), verbose=0)
    pred_idx = np.argmax(pred)
    pred_label = class_names[pred_idx]
    confidence = np.max(pred)

    plt.imshow(img)
    plt.title(f"Vrai: {true_label}\nPrédit: {pred_label} ({confidence:.2f})")
    plt.axis('off')

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/plantes_dataset/predictions_examples.png')
plt.show()

print("\n✅ Tâches terminées avec succès!")

In [ ]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.preprocessing import image
from tensorflow.keras.models import load_model
import matplotlib.pyplot as plt
import os
from google.colab import drive

# Monter Google Drive (important!)
drive.mount('/content/drive')

# ================= CONFIGURATION =================
model_path = "/content/drive/MyDrive/plantes_dataset/best_model.h5"
class_names = ["Brulure", "Rouille_Commune", "Sain", "Tache_grise",
               "Pourriture_charbonnee", "Pourriture_petiole", "Tige_saine"]

# Classes considérées comme saines
healthy_classes = ["Sain", "Tige_saine"]

# ================= CHARGEMENT DU MODÈLE =================
model = load_model(model_path)
print("✅ Modèle chargé avec succès")

# ================= FONCTION DE PRÉDICTION =================
def predict_plant_health(image_path, show_image=True):
    """
    Prédit si une plante est saine ou malade à partir d'une image
    Retourne:
        - État de santé ("Sain" ou "Malade")
        - Classe spécifique prédite
        - Niveau de confiance
    """
    # Charger et prétraiter l'image
    img = image.load_img(image_path, target_size=(224, 224))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0) / 255.0

    # Faire la prédiction
    predictions = model.predict(img_array)
    class_idx = np.argmax(predictions[0])
    confidence = np.max(predictions[0])
    specific_class = class_names[class_idx]

    # Déterminer l'état de santé général
    health_status = "Sain" if specific_class in healthy_classes else "Malade"

    # Créer une visualisation
    if show_image:
        plt.figure(figsize=(8, 8))
        plt.imshow(img)

        # Créer le texte du titre
        title_text = f"État: {health_status}\nClasse: {specific_class} ({confidence:.2%})"

        # Appliquer la couleur appropriée
        title_color = 'green' if health_status == "Sain" else 'red'
        plt.title(title_text, color=title_color)

        plt.axis('off')
        plt.show()

    return health_status, specific_class, confidence

# ================= UTILISATION =================
if __name__ == "__main__":
    # Chemin vers une image de test (à remplacer par votre image)
    test_image_path = "/content/drive/MyDrive/plantes_dataset/test_1.jfif"

    # Vérifier si l'image existe
    if not os.path.exists(test_image_path):
        print("❌ Image de test introuvable. Veuillez télécharger une image.")
        from google.colab import files
        uploaded = files.upload()
        test_image_path = next(iter(uploaded.keys()))

    # Faire la prédiction
    health_status, specific_class, confidence = predict_plant_health(test_image_path)

    # Afficher les résultats
    print("\n" + "="*50)
    print("RÉSULTATS DE LA PRÉDICTION")
    print("="*50)
    print(f"État de santé: {health_status}")
    print(f"Diagnostic spécifique: {specific_class}")
    print(f"Niveau de confiance: {confidence:.2%}")
    print("\n" + "="*50)

    # Interprétation supplémentaire
    if health_status == "Sain":
        print("✅ Votre plante semble en bonne santé!")
    else:
        print("⚠️ Votre plante montre des signes de maladie!")
        print(f"Type détecté: {specific_class}")
        print("Consultez un expert agricole pour des conseils de traitement.")